In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS dbw_kroger_insights.sandbox.landing")

base = "/Volumes/dbw_kroger_insights/sandbox/landing"
dbutils.fs.put(f"{base}/orders/day1.csv",
               "order_id,product_id,qty\n1,1005182,2\n2,1005175,1\n3,1002099,4\n",
               overwrite=True)
display(dbutils.fs.ls(f"{base}/orders"))

In [0]:
(spark.readStream
    .format("cloudFiles")                                   # = Auto Loader
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", f"{base}/_schema/orders")
    .load(f"{base}/orders")
 .writeStream
    .option("checkpointLocation", f"{base}/_checkpoint/orders")
    .trigger(availableNow=True)                             # process what's new, then stop
    .toTable("dbw_kroger_insights.sandbox.orders_bronze")
 .awaitTermination())

display(spark.sql("SELECT COUNT(*) AS rows FROM dbw_kroger_insights.sandbox.orders_bronze"))

In [0]:
dbutils.fs.put(f"{base}/orders/day2.csv",
               "order_id,product_id,qty\n4,99000001,1\n5,99000002,3\n",
               overwrite=True)

In [0]:
display(spark.sql("DESCRIBE HISTORY dbw_kroger_insights.sandbox.orders_bronze")
        .select("version", "operation", "operationMetrics.numOutputRows"))

In [0]:
display(dbutils.fs.ls("/Volumes/dbw_kroger_insights/sandbox/landing/orders"))

In [0]:
display(spark.sql("DESCRIBE HISTORY dbw_kroger_insights.sandbox.orders_bronze")
        .select("version", "operation", "operationMetrics.numOutputRows"))
display(spark.sql("SELECT COUNT(*) AS rows FROM dbw_kroger_insights.sandbox.orders_bronze"))

In [0]:
%sql
DROP SCHEMA dbw_kroger_insights.sandbox CASCADE;